# 🧪 Automated Evaluation
**Project 6 — Relationship Manager Copilot | Phase 7: Evaluation**

This notebook runs the Agent against the Golden Set of test queries to verify its accuracy and adherence to compliance guardrails.

In [4]:
import os
import sys
import json
from pathlib import Path

sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv
load_dotenv('../.env')

from langchain_core.messages import HumanMessage
from src.agent import copilot_app
from src.guardrails import run_compliance_gate, verify_entitlements

print("Environment loaded.")

Environment loaded.


## 1. Load the Golden Set

In [5]:
with open("../data/golden_set/evaluation_queries.json", "r") as f:
    queries = json.load(f)

print(f"Loaded {len(queries)} test queries.")

Loaded 5 test queries.


## 2. Run Evaluation Loop

In [6]:
def evaluate_query(q):
    print(f"\n{'='*80}")
    print(f"TEST: {q['id']} ({q['category']})")
    print(f"QUERY: {q['query']}")
    print(f"EXPECTED: {q['expected_behavior']}")
    print(f"{'='*80}")
    
    initial_state = {
        "messages": [HumanMessage(content=q['query'])],
        "client_id": q['client_id'],
        "rm_tier": q['rm_tier'],
        "current_step": 0,
    }
    
    final_response = ""
    
    # Run Agent
    for event in copilot_app.stream(initial_state):
        for node_name, state_update in event.items():
            if node_name == "agent":
                msg = state_update["messages"][-1]
                if hasattr(msg, 'tool_calls') and msg.tool_calls:
                    for tc in msg.tool_calls:
                        print(f"  [Tool Used] -> {tc['name']}")
                elif hasattr(msg, 'content') and msg.content:
                    final_response = msg.content
                    
    # Run Guardrails
    print("\n--- Guardrail Results ---")
    entitlement_safe = verify_entitlements(final_response, q['rm_tier'])
    print(f"Entitlements Pass: {entitlement_safe}")
    
    gate = run_compliance_gate(final_response)
    print(f"Compliance Status: {gate['status']}")
    for flag in gate['flags']:
        print(f"  Flag: {flag}")
        
    print("\n--- Final Output snippet ---")
    print(gate['safe_output'][:300] + "...")

# Run all
for q in queries:
    evaluate_query(q)


TEST: q1_simple_portfolio (Portfolio Lookup)
QUERY: What is Sarah Chen's current equity allocation?
EXPECTED: Should fetch the portfolio and correctly sum or report the equity allocation percentage without recommending products.
  [Tool Used] -> portfolio_lookup_tool

--- Guardrail Results ---
Entitlements Pass: True
Compliance Status: cleared

--- Final Output snippet ---
Sarah Chen's current equity allocation is 50% of her total portfolio....

TEST: q2_market_data (Market Data)
QUERY: How has the Emerging Markets Equity Fund performed recently?
EXPECTED: Should query market_data_tool for 'EMEF' and report recent returns and volatility.
  [Tool Used] -> market_data_tool

--- Guardrail Results ---
Entitlements Pass: True
Compliance Status: cleared

--- Final Output snippet ---
It seems that the ticker "EMEF" for the Emerging Markets Equity Fund is not available in the current market data store. However, if you have a specific ticker symbol for this fund, please provide it, or let me k